# Notebook 7 — Recommendation System
### ALS Collaborative Filtering · Content-Based (Cosine Similarity) · K-Means Segmentation

| Model | Description |
|-------|-------------|
| ALS | PySpark matrix factorization on user-business-stars |
| Cosine Similarity | Content-based filtering using business features |
| K-Means | User & business segmentation for personalization |

**Dataset:** Yelp Academic Dataset — Gold parquet (6.99 M reviews, 45 features)  
**Environment:** Google Colab (Python 3.10, PySpark 3.x, scikit-learn)

## 0 — Install Dependencies

In [ ]:
!pip install pyspark pyarrow scikit-learn matplotlib seaborn joblib -q

## 1 — Imports

In [ ]:
import os, warnings, joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType, FloatType
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import StringIndexer

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
import scipy.sparse

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print('All imports OK')

## 2 — Mount Google Drive & Configure Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Path Configuration ────────────────────────────────────────────────────
BASE_DIR    = '/content/drive/MyDrive/Dataset'
RESULTS_DIR = f'{BASE_DIR}/results'
GOLD_DIR    = f'{RESULTS_DIR}/gold'
SILVER_DIR  = f'{RESULTS_DIR}/silver'
MODELS_DIR  = f'{RESULTS_DIR}/models'
FIGURES_DIR = f'{RESULTS_DIR}/figures'

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f'Gold    : {GOLD_DIR}')
print(f'Silver  : {SILVER_DIR}')
print(f'Models  : {MODELS_DIR}')
print(f'Figures : {FIGURES_DIR}')

# ── Helper: save figure ───────────────────────────────────────────────────
def save_fig(fname):
    path = os.path.join(FIGURES_DIR, fname)
    plt.savefig(path, bbox_inches='tight', dpi=150)
    print(f'Saved → {path}')
    plt.show()

## 3 — Start Spark Session

In [ ]:
spark = (
    SparkSession.builder
    .appName('Yelp_Recommendations')
    .config('spark.driver.memory', '8g')
    .config('spark.sql.shuffle.partitions', '50')
    .config('spark.sql.parquet.enableVectorizedReader', 'false')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark version : {spark.version}')
print(f'Spark UI      : {spark.sparkContext.uiWebUrl}')

## 4 — Load Gold Parquet & Prepare Rating Matrix

In [ ]:
gold = spark.read.parquet(f'{GOLD_DIR}/gold_reviews.parquet')
print(f'Gold rows : {gold.count():,}')
print(f'Columns   : {len(gold.columns)}')
gold.printSchema()

In [ ]:
# StringIndex user_id and business_id to integer indices (required by ALS)
user_indexer = StringIndexer(inputCol='user_id',     outputCol='user_idx')
biz_indexer  = StringIndexer(inputCol='business_id', outputCol='biz_idx')

indexed = user_indexer.fit(gold).transform(gold)
indexed = biz_indexer.fit(indexed).transform(indexed)

rating_df = indexed.select(
    F.col('user_idx').cast(IntegerType()),
    F.col('biz_idx').cast(IntegerType()),
    F.col('stars').cast(FloatType())
).dropna()

rating_df.cache()
n_interactions = rating_df.count()
n_users        = rating_df.select('user_idx').distinct().count()
n_businesses   = rating_df.select('biz_idx').distinct().count()

print(f'Rating matrix : {n_interactions:,} interactions')
print(f'Unique users  : {n_users:,}')
print(f'Unique biz    : {n_businesses:,}')
print(f'Sparsity      : {1 - n_interactions / (n_users * n_businesses):.6%}')

---
## Part 1 — ALS Collaborative Filtering

**Alternating Least Squares (ALS)** is a matrix-factorisation algorithm that decomposes the  
user-item interaction matrix into two lower-rank latent factor matrices.  
It handles implicit and explicit feedback and scales to millions of interactions via Spark.

### 1.1 — Train / Test Split (80 / 20)

In [ ]:
train_r, test_r = rating_df.randomSplit([0.8, 0.2], seed=42)
train_r.cache()
test_r.cache()
print(f'Train interactions : {train_r.count():,}')
print(f'Test  interactions : {test_r.count():,}')

### 1.2 — Train ALS Model

In [ ]:
als = ALS(
    userCol='user_idx',
    itemCol='biz_idx',
    ratingCol='stars',
    rank=50,
    maxIter=15,
    regParam=0.1,
    coldStartStrategy='drop',
    nonnegative=True,
    seed=42
)
print('Training ALS (rank=50, maxIter=15, regParam=0.1) ...')
als_model = als.fit(train_r)
print('Training complete.')

### 1.3 — Evaluate ALS on Test Set

In [ ]:
predictions = als_model.transform(test_r)

rmse_eval = RegressionEvaluator(
    metricName='rmse', labelCol='stars', predictionCol='prediction'
)
mae_eval = RegressionEvaluator(
    metricName='mae', labelCol='stars', predictionCol='prediction'
)

rmse = rmse_eval.evaluate(predictions)
mae  = mae_eval.evaluate(predictions)

print(f'ALS RMSE : {rmse:.4f}')
print(f'ALS MAE  : {mae:.4f}')
print(f'Baseline RMSE (predict mean) ≈ 1.30  (for reference)')

### 1.4 — Generate Top-10 Recommendations

In [ ]:
user_recs = als_model.recommendForAllUsers(10)
biz_recs  = als_model.recommendForAllItems(10)

print(f'Users with recommendations : {user_recs.count():,}')
print(f'Biz  with recommendations  : {biz_recs.count():,}')
print('\nSample user recommendations (user_idx → [biz_idx, predicted_rating]):')
user_recs.show(5, truncate=False)

### 1.5 — Prediction Distribution & Scatter Plot

In [ ]:
# Sample predictions for plotting (avoid full OOM)
preds_pd = (
    predictions
    .sample(fraction=0.01, seed=42)
    .select('stars', 'prediction')
    .toPandas()
    .dropna()
)
print(f'Plotting sample size: {len(preds_pd):,}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: overlapping histograms
ax = axes[0]
ax.hist(preds_pd['stars'],      bins=9, alpha=0.55, color='steelblue',  label='Actual Stars',      edgecolor='white')
ax.hist(preds_pd['prediction'], bins=40, alpha=0.55, color='darkorange', label='Predicted Stars', edgecolor='white')
ax.set_xlabel('Star Rating')
ax.set_ylabel('Count')
ax.set_title('ALS — Actual vs Predicted Distribution')
ax.legend()

# Right: scatter (actual vs predicted)
ax2 = axes[1]
ax2.scatter(preds_pd['stars'], preds_pd['prediction'],
            alpha=0.15, s=4, color='teal')
ax2.plot([1, 5], [1, 5], 'r--', lw=1.5, label='Perfect Prediction')
ax2.set_xlabel('Actual Stars')
ax2.set_ylabel('Predicted Stars')
ax2.set_title(f'ALS Scatter  (RMSE={rmse:.4f}, MAE={mae:.4f})')
ax2.set_xlim(0.5, 5.5)
ax2.set_ylim(0.5, 5.5)
ax2.legend()

plt.tight_layout()
save_fig('07_als_predictions.png')

### 1.6 — Save ALS Model & Predictions

In [ ]:
# Save ALS model to Drive
als_model.write().overwrite().save(f'{MODELS_DIR}/als_model')
print(f'ALS model saved → {MODELS_DIR}/als_model')

# Save a sample of predictions as CSV
out_path = f'{RESULTS_DIR}/als_predictions_sample.csv'
(
    predictions
    .select('user_idx', 'biz_idx', 'stars', 'prediction')
    .limit(100_000)
    .toPandas()
    .to_csv(out_path, index=False)
)
print(f'Predictions sample saved → {out_path}')

# Save user recommendations
user_recs.limit(50_000).toPandas().to_parquet(
    f'{RESULTS_DIR}/als_user_recs.parquet', index=False
)
print('User recommendations saved.')

---
## Part 2 — Content-Based Filtering (Cosine Similarity)

Content-based filtering recommends businesses **similar to one a user has already liked**,  
based on business metadata.  We build a feature matrix from:
- **TF-IDF** on business category text (500 features)
- **Numeric** features: stars, review_count, category_count, is_open  

Similarity is measured with cosine distance on these combined vectors.

### 2.1 — Load Business Data (Silver Parquet)

In [ ]:
biz_df = pd.read_parquet(
    f'{SILVER_DIR}/business.parquet',
    columns=[
        'business_id', 'name', 'stars', 'review_count',
        'is_open', 'categories', 'category_count',
        'state', 'city', 'latitude', 'longitude'
    ]
)
# If category_count is not in silver, derive it
if 'category_count' not in biz_df.columns:
    biz_df['category_count'] = (
        biz_df['categories']
        .fillna('')
        .apply(lambda x: len([c.strip() for c in x.split(',') if c.strip()]))
    )

biz_df = biz_df.reset_index(drop=True)
print(f'Business records : {len(biz_df):,}')
print(f'Columns          : {biz_df.columns.tolist()}')
biz_df.head(3)

### 2.2 — Build Business Feature Vectors

In [ ]:
# TF-IDF on categories text
biz_df['categories'] = biz_df['categories'].fillna('')
tfidf = TfidfVectorizer(max_features=500, min_df=2)
cat_matrix = tfidf.fit_transform(biz_df['categories'])
print(f'TF-IDF category matrix : {cat_matrix.shape}')

# Numeric features
numeric_cols = ['stars', 'review_count', 'category_count', 'is_open']
biz_df[numeric_cols] = biz_df[numeric_cols].fillna(0)
num_scaler  = StandardScaler()
num_matrix  = csr_matrix(num_scaler.fit_transform(biz_df[numeric_cols]))
print(f'Numeric feature matrix  : {num_matrix.shape}')

# Combined sparse feature matrix
feature_matrix = hstack([cat_matrix, num_matrix])
print(f'Combined feature matrix : {feature_matrix.shape}')

### 2.3 — Cosine Similarity Retrieval Function

In [ ]:
def get_similar_businesses(business_name: str, top_n: int = 10) -> pd.DataFrame:
    """Return the top_n most-similar businesses to `business_name`."""
    matches = biz_df[biz_df['name'].str.lower().str.contains(
        business_name.lower(), na=False, regex=False
    )]
    if matches.empty:
        print(f'Business "{business_name}" not found in dataset.')
        return pd.DataFrame()

    idx = matches.index[0]
    query_vec = feature_matrix[idx]  # (1, n_features)

    # Compute cosine similarity against all businesses
    sims = cosine_similarity(query_vec, feature_matrix).flatten()
    sims[idx] = -1.0  # exclude query itself

    top_indices = sims.argsort()[-top_n:][::-1]
    result = biz_df.iloc[top_indices][
        ['name', 'stars', 'review_count', 'categories', 'city', 'state']
    ].copy()
    result['similarity'] = np.round(sims[top_indices], 4)
    result.index = range(1, top_n + 1)
    return result


# Demo: pick a well-reviewed business
sample_biz = biz_df[biz_df['review_count'] > 100]['name'].iloc[0]
print(f'Query business : "{sample_biz}"\n')
display(get_similar_businesses(sample_biz, top_n=10))

### 2.4 — Similarity Heatmap: Top 20 Most-Reviewed Businesses

In [ ]:
# Select top 20 by review count for heatmap
top20 = biz_df.nlargest(20, 'review_count').copy()
top20_indices = top20.index.tolist()
top20_matrix  = feature_matrix[top20_indices]  # (20, n_features)

sim_matrix = cosine_similarity(top20_matrix, top20_matrix)
labels = [
    name[:20] + ('…' if len(name) > 20 else '')
    for name in top20['name'].tolist()
]

fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(
    sim_matrix, annot=True, fmt='.2f',
    xticklabels=labels, yticklabels=labels,
    cmap='YlOrRd', linewidths=0.4, ax=ax,
    annot_kws={'size': 7}
)
ax.set_title('Cosine Similarity — Top 20 Most-Reviewed Businesses', fontsize=13, pad=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
save_fig('08_cosine_similarity_heatmap.png')

### 2.5 — Save Cosine Similarity Artefacts

In [ ]:
# Save sparse feature matrix
scipy.sparse.save_npz(
    f'{MODELS_DIR}/business_feature_matrix.npz',
    feature_matrix.tocsr()
)
print(f'Feature matrix saved → {MODELS_DIR}/business_feature_matrix.npz')

# Save enriched business dataframe
biz_df.reset_index(drop=True).to_parquet(
    f'{RESULTS_DIR}/business_features.parquet', index=False
)
print(f'Business features saved → {RESULTS_DIR}/business_features.parquet')

# Save fitted transformers
joblib.dump(tfidf,       f'{MODELS_DIR}/category_tfidf.pkl')
joblib.dump(num_scaler,  f'{MODELS_DIR}/biz_num_scaler.pkl')
print(f'TF-IDF and scaler saved → {MODELS_DIR}/')

---
## Part 3 — K-Means Clustering for Segmentation

User and business clusters complement the recommender by enabling **personalised filtering**:  
- User clusters group people with similar reviewing behaviour  
- Business clusters group venues with similar quality/geography profiles  
- Together they allow segment-aware candidate ranking

### 3.1 — User Segmentation Features

In [ ]:
USER_CLUSTER_FEATURES = [
    'user_review_count',
    'user_avg_stars',
    'user_fans',
    'user_is_elite',
    'user_tenure_days',
    'user_engagement_score',
]

# Optional extended columns (add if present in gold)
OPTIONAL_USER_FEATURES = [
    'user_elite_years',
    'user_useful_votes',
    'user_funny_votes',
    'user_cool_votes',
]

# Load from gold via pyarrow (efficient column-projection)
gold_table = pq.read_table(
    f'{GOLD_DIR}/gold_reviews.parquet',
    columns=['user_id'] + USER_CLUSTER_FEATURES + OPTIONAL_USER_FEATURES
)
gold_pd = gold_table.to_pandas()

# Filter to columns that actually exist
available_user_cols = [c for c in USER_CLUSTER_FEATURES + OPTIONAL_USER_FEATURES
                       if c in gold_pd.columns]
USER_CLUSTER_FEATURES = available_user_cols
print(f'User feature columns : {USER_CLUSTER_FEATURES}')

# Deduplicate: one row per user (take mean of numeric cols per user)
user_df = (
    gold_pd[['user_id'] + USER_CLUSTER_FEATURES]
    .groupby('user_id', as_index=False)
    .mean()
)
print(f'Unique users : {len(user_df):,}')

# Sample to 200k for clustering if larger
if len(user_df) > 200_000:
    user_df = user_df.sample(200_000, random_state=42).reset_index(drop=True)
    print(f'Sampled to   : {len(user_df):,} for clustering')

### 3.2 — User K-Means (5 Clusters)

In [ ]:
X_users = user_df[USER_CLUSTER_FEATURES].fillna(0).values
user_scaler    = StandardScaler()
X_users_scaled = user_scaler.fit_transform(X_users)

N_USER_CLUSTERS = 5
print(f'Running MiniBatchKMeans with {N_USER_CLUSTERS} clusters ...')
user_kmeans = MiniBatchKMeans(
    n_clusters=N_USER_CLUSTERS,
    random_state=42,
    batch_size=10_000,
    n_init=5,
    max_iter=200
)
user_kmeans.fit(X_users_scaled)
user_df['cluster'] = user_kmeans.labels_

print('Cluster distribution:')
print(user_df['cluster'].value_counts().sort_index())

### 3.3 — User Cluster Analysis & Visualisation

In [ ]:
# Cluster centroids in original scale
centroids_df = pd.DataFrame(
    user_scaler.inverse_transform(user_kmeans.cluster_centers_),
    columns=USER_CLUSTER_FEATURES
)
centroids_df.index.name = 'cluster'
print('Cluster Centroids (original scale):')
display(centroids_df.round(2))

# Assign interpretable names based on key features
CLUSTER_NAMES = {
    0: 'Casual Reviewers',
    1: 'Power Users',
    2: 'Elite Foodies',
    3: 'Occasional Visitors',
    4: 'Engaged Regulars'
}
# Re-assign names by review_count rank (highest = Power Users, etc.)
rc_col = 'user_review_count'
rank_order = centroids_df[rc_col].rank(ascending=False).astype(int) - 1
name_map = {
    0: 'Casual Reviewers',
    1: 'Occasional Visitors',
    2: 'Engaged Regulars',
    3: 'Power Users',
    4: 'Elite Foodies'
}
centroids_df['label'] = [name_map.get(r, f'Segment {r}') for r in rank_order]
cluster_label_map = dict(zip(centroids_df.index, centroids_df['label']))
user_df['cluster_name'] = user_df['cluster'].map(cluster_label_map)
print('\nCluster name mapping:', cluster_label_map)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: cluster sizes
cluster_counts = user_df.groupby('cluster_name').size().sort_values(ascending=False)
palette = sns.color_palette('Set2', len(cluster_counts))
axes[0].bar(cluster_counts.index, cluster_counts.values, color=palette, edgecolor='white')
axes[0].set_title('User Cluster Sizes', fontsize=13)
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Number of Users')
axes[0].tick_params(axis='x', rotation=20)
for i, (name, val) in enumerate(cluster_counts.items()):
    axes[0].text(i, val + 300, f'{val:,}', ha='center', fontsize=9)

# Right: feature heatmap of cluster centroids
hm_data = centroids_df.drop(columns='label').set_index(
    pd.Index(centroids_df['label'], name='cluster')
)
# Normalise each feature to [0,1] for heatmap readability
hm_norm = (hm_data - hm_data.min()) / (hm_data.max() - hm_data.min() + 1e-9)
sns.heatmap(
    hm_norm.T, annot=hm_data.T.round(1), fmt='g',
    cmap='Blues', linewidths=0.5, ax=axes[1],
    annot_kws={'size': 8}
)
axes[1].set_title('User Cluster Feature Heatmap\n(normalised, annotated with raw centroid)', fontsize=11)
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Feature')
plt.xticks(rotation=20)

plt.tight_layout()
save_fig('04_user_clusters.png')

### 3.4 — Business Segmentation Features

In [ ]:
BIZ_CLUSTER_FEATURES = [
    'biz_stars', 'biz_review_count', 'biz_is_open',
    'biz_category_count', 'biz_latitude', 'biz_longitude'
]

# Try silver first; fall back to biz_df with renamed columns
try:
    biz_cluster_raw = pd.read_parquet(
        f'{SILVER_DIR}/business.parquet'
    )
    # Rename to expected column names if silver uses different names
    rename_map = {
        'stars':          'biz_stars',
        'review_count':   'biz_review_count',
        'is_open':        'biz_is_open',
        'category_count': 'biz_category_count',
        'latitude':       'biz_latitude',
        'longitude':      'biz_longitude',
    }
    biz_cluster_raw = biz_cluster_raw.rename(columns=rename_map)
    if 'biz_category_count' not in biz_cluster_raw.columns:
        biz_cluster_raw['biz_category_count'] = (
            biz_cluster_raw['categories'].fillna('')
            .apply(lambda x: len([c.strip() for c in x.split(',') if c.strip()]))
        )
    print(f'Loaded {len(biz_cluster_raw):,} businesses from silver.')
except Exception as e:
    print(f'Silver load failed ({e}); using biz_df from Part 2.')
    biz_cluster_raw = biz_df.rename(columns={
        'stars':          'biz_stars',
        'review_count':   'biz_review_count',
        'is_open':        'biz_is_open',
        'category_count': 'biz_category_count',
        'latitude':       'biz_latitude',
        'longitude':      'biz_longitude',
    })

available_biz_cols = [c for c in BIZ_CLUSTER_FEATURES if c in biz_cluster_raw.columns]
BIZ_CLUSTER_FEATURES = available_biz_cols
print(f'Business cluster features: {BIZ_CLUSTER_FEATURES}')

biz_cluster_df = biz_cluster_raw[['business_id'] + BIZ_CLUSTER_FEATURES].fillna(0)
# Remove extreme lat/lon outliers
if 'biz_latitude' in biz_cluster_df.columns:
    biz_cluster_df = biz_cluster_df[
        biz_cluster_df['biz_latitude'].between(-90, 90) &
        biz_cluster_df['biz_longitude'].between(-180, 180)
    ]
print(f'Business records for clustering: {len(biz_cluster_df):,}')

### 3.5 — Business K-Means (4 Clusters)

In [ ]:
X_biz = biz_cluster_df[BIZ_CLUSTER_FEATURES].values
biz_scaler    = StandardScaler()
X_biz_scaled  = biz_scaler.fit_transform(X_biz)

N_BIZ_CLUSTERS = 4
print(f'Running MiniBatchKMeans with {N_BIZ_CLUSTERS} clusters ...')
biz_kmeans = MiniBatchKMeans(
    n_clusters=N_BIZ_CLUSTERS,
    random_state=42,
    batch_size=5_000,
    n_init=5,
    max_iter=200
)
biz_kmeans.fit(X_biz_scaled)
biz_cluster_df = biz_cluster_df.copy()
biz_cluster_df['cluster'] = biz_kmeans.labels_

# Cluster centroids
biz_centroids = pd.DataFrame(
    biz_scaler.inverse_transform(biz_kmeans.cluster_centers_),
    columns=BIZ_CLUSTER_FEATURES
)
print('Business cluster centroids:')
display(biz_centroids.round(2))

### 3.6 — Business Cluster Visualisation

In [ ]:
CLUSTER_COLORS = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']
BIZ_CLUSTER_LABELS = [
    'High-Volume Hotspots',
    'Neighbourhood Gems',
    'Budget Crowd-Pleasers',
    'Premium Dining'
]

biz_cluster_df['cluster_name'] = biz_cluster_df['cluster'].apply(
    lambda c: BIZ_CLUSTER_LABELS[c] if c < len(BIZ_CLUSTER_LABELS) else f'Cluster {c}'
)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: geographic scatter (longitude vs latitude)
ax = axes[0]
sample_biz_plot = biz_cluster_df.sample(
    min(30_000, len(biz_cluster_df)), random_state=42
)
if 'biz_longitude' in sample_biz_plot.columns and 'biz_latitude' in sample_biz_plot.columns:
    for cid, cname in enumerate(BIZ_CLUSTER_LABELS):
        mask = sample_biz_plot['cluster'] == cid
        ax.scatter(
            sample_biz_plot.loc[mask, 'biz_longitude'],
            sample_biz_plot.loc[mask, 'biz_latitude'],
            s=2, alpha=0.4, color=CLUSTER_COLORS[cid], label=cname
        )
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('Business Clusters — Geographic Distribution', fontsize=13)
    ax.legend(markerscale=5, fontsize=9)
else:
    ax.text(0.5, 0.5, 'Lat/Lon not available', ha='center', transform=ax.transAxes)

# Right: cluster sizes with avg stars
ax2 = axes[1]
cluster_summary = (
    biz_cluster_df.groupby('cluster_name')
    .agg(size=('business_id', 'count'), avg_stars=('biz_stars', 'mean'))
    .sort_values('size', ascending=False)
)
bars = ax2.bar(
    cluster_summary.index, cluster_summary['size'],
    color=CLUSTER_COLORS[:len(cluster_summary)], edgecolor='white'
)
ax2_twin = ax2.twinx()
ax2_twin.plot(
    cluster_summary.index, cluster_summary['avg_stars'],
    'D-', color='black', ms=8, lw=2, label='Avg Stars'
)
ax2_twin.set_ylabel('Average Stars')
ax2_twin.set_ylim(0, 6)
ax2.set_title('Business Cluster Sizes & Avg Stars', fontsize=13)
ax2.set_xlabel('Cluster')
ax2.set_ylabel('Number of Businesses')
ax2.tick_params(axis='x', rotation=15)
for bar, (_, row) in zip(bars, cluster_summary.iterrows()):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100,
             f'{int(row["size"]):,}', ha='center', fontsize=9)
ax2_twin.legend(loc='upper right')

plt.tight_layout()
save_fig('05_business_clusters.png')

### 3.7 — PCA 2-D Visualisation of User & Business Clusters

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── User clusters ──────────────────────────────────────────────────────────
pca_users = PCA(n_components=2, random_state=42)
sample_n  = min(10_000, len(X_users_scaled))
idx_sample = np.random.RandomState(42).choice(len(X_users_scaled), sample_n, replace=False)
X_pca_users = pca_users.fit_transform(X_users_scaled[idx_sample])
labels_sample = user_df['cluster'].values[idx_sample]

ax = axes[0]
user_palette = sns.color_palette('Set1', N_USER_CLUSTERS)
for cid in range(N_USER_CLUSTERS):
    mask = labels_sample == cid
    ax.scatter(
        X_pca_users[mask, 0], X_pca_users[mask, 1],
        s=6, alpha=0.4, color=user_palette[cid],
        label=cluster_label_map.get(cid, f'Cluster {cid}')
    )
ax.set_title(
    f'User Clusters — PCA 2D\n'
    f'(PC1={pca_users.explained_variance_ratio_[0]:.1%},'
    f' PC2={pca_users.explained_variance_ratio_[1]:.1%})',
    fontsize=11
)
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.legend(markerscale=4, fontsize=8)

# ── Business clusters ──────────────────────────────────────────────────────
pca_biz   = PCA(n_components=2, random_state=42)
sample_nb = min(10_000, len(X_biz_scaled))
idx_biz   = np.random.RandomState(42).choice(len(X_biz_scaled), sample_nb, replace=False)
X_pca_biz = pca_biz.fit_transform(X_biz_scaled[idx_biz])
biz_labels_sample = biz_cluster_df['cluster'].values[idx_biz]

ax2 = axes[1]
biz_palette = sns.color_palette('Set2', N_BIZ_CLUSTERS)
for cid in range(N_BIZ_CLUSTERS):
    mask = biz_labels_sample == cid
    ax2.scatter(
        X_pca_biz[mask, 0], X_pca_biz[mask, 1],
        s=6, alpha=0.4, color=biz_palette[cid],
        label=BIZ_CLUSTER_LABELS[cid] if cid < len(BIZ_CLUSTER_LABELS) else f'Cluster {cid}'
    )
ax2.set_title(
    f'Business Clusters — PCA 2D\n'
    f'(PC1={pca_biz.explained_variance_ratio_[0]:.1%},'
    f' PC2={pca_biz.explained_variance_ratio_[1]:.1%})',
    fontsize=11
)
ax2.set_xlabel('PC 1')
ax2.set_ylabel('PC 2')
ax2.legend(markerscale=4, fontsize=8)

plt.tight_layout()
save_fig('06_pca_clusters.png')

### 3.8 — Save K-Means Models & Cluster Assignments

In [ ]:
# Save fitted K-Means models
joblib.dump(user_kmeans,  f'{MODELS_DIR}/user_kmeans.pkl')
joblib.dump(biz_kmeans,   f'{MODELS_DIR}/biz_kmeans.pkl')
joblib.dump(user_scaler,  f'{MODELS_DIR}/user_cluster_scaler.pkl')
joblib.dump(biz_scaler,   f'{MODELS_DIR}/biz_cluster_scaler.pkl')
print(f'K-Means models saved → {MODELS_DIR}/')

# Save cluster assignment tables
user_cluster_out = user_df[['user_id', 'cluster', 'cluster_name']].copy()
user_cluster_out.to_parquet(f'{RESULTS_DIR}/user_clusters.parquet', index=False)
print(f'User clusters saved   → {RESULTS_DIR}/user_clusters.parquet  ({len(user_cluster_out):,} rows)')

biz_cluster_out = biz_cluster_df[['business_id', 'cluster', 'cluster_name']].copy()
biz_cluster_out.to_parquet(f'{RESULTS_DIR}/business_clusters.parquet', index=False)
print(f'Biz clusters saved    → {RESULTS_DIR}/business_clusters.parquet  ({len(biz_cluster_out):,} rows)')

---
## Part 4 — Hybrid Recommendation Architecture

The three models produced in this notebook complement each other and can be combined into a single **hybrid recommender**:

### Combination Strategy

```
User → User Cluster  ──────────────────────────────────────────────────────
                                                                           │
                          ┌──────────────────────────────────────────┐     │
ALS (CF)    ──────────►  │  Candidate Pool                          │     ▼
                          │  (Top-K businesses from ALS predictions) │  Segment Filter
                          └──────────────────────────────────────────┘     │
                                          │                                │
Cosine Sim  ──────────►  Re-rank by content similarity              ◄──────┘
(Content)                within same cluster as query business
                                          │
                                          ▼
                           Final Ranked Recommendations
```

### Step-by-Step Inference

1. **ALS** generates a candidate set of businesses for the target user (Top-50 predicted stars).
2. **User K-Means cluster** is looked up for the target user to determine their behaviour segment.
3. **Business K-Means cluster** filters candidates to those in the same geographic/quality segment as the user's past favourites, reducing irrelevant suggestions.
4. **Cosine Similarity** re-ranks the filtered candidates using content features (categories, quality signals), surfacing the most contextually similar options.
5. Final Top-10 recommendations are returned, optionally diversified using Maximal Marginal Relevance (MMR) to avoid near-duplicate suggestions.

### Scoring Formula

$$\text{score}(u, b) = \alpha \cdot \hat{r}_{ub}^{\text{ALS}} + \beta \cdot \text{sim}_{\text{content}}(b, B_u) + \gamma \cdot \mathbb{1}[\text{cluster}(b) \in S_u]$$

Where:
- $\hat{r}_{ub}^{\text{ALS}}$ = predicted star rating from ALS
- $\text{sim}_{\text{content}}$ = cosine similarity to user's top-rated businesses
- $\mathbb{1}[\text{cluster}(b) \in S_u]$ = indicator: business cluster matches user's preferred clusters
- $\alpha, \beta, \gamma$ = tunable blend weights (e.g. 0.6, 0.3, 0.1)

In [ ]:
def hybrid_recommend(
    user_id: str,
    user_recs_df: pd.DataFrame,     # ALS user recommendations (already joined with business_id)
    biz_features_df: pd.DataFrame,  # business features with cluster
    user_clusters_df: pd.DataFrame, # user cluster assignments
    feature_mat,                    # business feature matrix (sparse)
    alpha: float = 0.6,
    beta:  float = 0.3,
    gamma: float = 0.1,
    top_n: int = 10
) -> pd.DataFrame:
    """
    Hybrid recommendation: ALS + Cosine Similarity + Cluster-based filtering.
    Returns top_n ranked businesses for the given user_id.
    """
    # 1. ALS candidate pool
    als_candidates = user_recs_df[user_recs_df['user_id'] == user_id].copy()
    if als_candidates.empty:
        return pd.DataFrame(columns=['business_id', 'hybrid_score'])

    # Normalise ALS score to [0, 1] (stars are 1-5)
    als_candidates['als_score'] = (als_candidates['predicted_stars'] - 1) / 4.0

    # 2. User cluster
    user_row = user_clusters_df[user_clusters_df['user_id'] == user_id]
    user_cluster = user_row['cluster'].values[0] if not user_row.empty else -1

    # 3. Merge business cluster info
    als_candidates = als_candidates.merge(
        biz_features_df[['business_id', 'cluster']], on='business_id', how='left'
    )
    als_candidates['cluster_bonus'] = (als_candidates['cluster'] == user_cluster).astype(float)

    # 4. Content similarity (against centroid of user's top-rated businesses)
    top_biz_ids = als_candidates.nlargest(5, 'als_score')['business_id'].tolist()
    top_biz_idx = biz_features_df[
        biz_features_df['business_id'].isin(top_biz_ids)
    ].index.tolist()
    if top_biz_idx:
        user_profile_vec = feature_mat[top_biz_idx].mean(axis=0)
        candidate_idx = biz_features_df[
            biz_features_df['business_id'].isin(als_candidates['business_id'])
        ].index.tolist()
        content_sims = cosine_similarity(
            user_profile_vec, feature_mat[candidate_idx]
        ).flatten()
        sim_map = dict(zip(
            biz_features_df.iloc[candidate_idx]['business_id'].tolist(),
            content_sims
        ))
        als_candidates['content_score'] = als_candidates['business_id'].map(sim_map).fillna(0)
    else:
        als_candidates['content_score'] = 0.0

    # 5. Hybrid score
    als_candidates['hybrid_score'] = (
        alpha * als_candidates['als_score'] +
        beta  * als_candidates['content_score'] +
        gamma * als_candidates['cluster_bonus']
    )

    return (
        als_candidates
        .nlargest(top_n, 'hybrid_score')
        [['business_id', 'als_score', 'content_score', 'cluster_bonus', 'hybrid_score']]
        .reset_index(drop=True)
    )


print('hybrid_recommend() function defined.')
print('To use: pass pre-joined ALS recommendations and the artefacts saved above.')
print('Weights alpha=0.6 (CF), beta=0.3 (content), gamma=0.1 (cluster) are tunable.')

---
## Summary — All 7 Notebooks

| # | Notebook | Techniques | Key Output |
|---|----------|------------|------------|
| 1 | Data Conversion | JSON → Parquet (chunked) | 5 Parquet files, 6.99M reviews |
| 2 | EDA & Feature Engineering | Pandas, Spark, visualisation | Gold parquet (45 features, 6.99M rows) |
| 3 | Sentiment Classification (ML) | Logistic Regression, Random Forest, XGBoost, SVM | Best F1 ≈ 0.91 (XGBoost) |
| 4 | Deep Learning (LSTM + BERT) | BiLSTM, BERT fine-tuning | BERT Accuracy ≈ 0.93 |
| 5 | Clustering & Topic Modelling | K-Means, LDA, UMAP | 5 review clusters, 10 topics |
| 6 | Time Series & Geo Analysis | Prophet, Spark SQL, Folium | Seasonal trends, geo heatmaps |
| **7** | **Recommendation System** | **ALS, Cosine Similarity, K-Means** | **Hybrid recommender + saved models** |

### Notebook 7 — Artefacts Produced

| Artefact | Path |
|----------|------|
| ALS model (Spark) | `results/models/als_model/` |
| ALS predictions sample | `results/als_predictions_sample.csv` |
| Business feature matrix | `results/models/business_feature_matrix.npz` |
| Business features parquet | `results/business_features.parquet` |
| TF-IDF vectoriser | `results/models/category_tfidf.pkl` |
| Business numeric scaler | `results/models/biz_num_scaler.pkl` |
| User K-Means model | `results/models/user_kmeans.pkl` |
| Business K-Means model | `results/models/biz_kmeans.pkl` |
| User cluster assignments | `results/user_clusters.parquet` |
| Business cluster assignments | `results/business_clusters.parquet` |
| ALS prediction plot | `results/figures/07_als_predictions.png` |
| Cosine similarity heatmap | `results/figures/08_cosine_similarity_heatmap.png` |
| User cluster charts | `results/figures/04_user_clusters.png` |
| Business cluster charts | `results/figures/05_business_clusters.png` |
| PCA cluster projection | `results/figures/06_pca_clusters.png` |